# The deployed surrogate — how to use it

**Sub-task 3.8 · demo notebook.** The surrogate is a fast machine-learning model that imitates EnergyPlus
for the DOE medium office: for any design (21 inputs) at any site it predicts 10 outputs — annual heating
and cooling demand, energy use intensity (EUI), peak heating and cooling, and cooling of the five zones.

Phase 4 (optimiser) and Phase 5 (web service) use it only through `surrogate.registry`.
This notebook shows that front door: a training city, a new site from its weather file, a refused design and
a 100,000-design batch. Decisions behind it: `claude/Phase3.8_Execution_Plan.md` (J1–J7).
Official speed numbers: `surrogate/outputs/registry/registry_report.md` (this notebook's timings are illustrative).

In [1]:
import time
import numpy as np
import pandas as pd
from surrogate import registry as R
from surrogate.registry import Surrogate, Site, DesignError
from surrogate.paths import EPW_DIR

pd.set_option("display.precision", 2)
pd.set_option("display.width", 160)

## 1. Load
`Surrogate.load()` reads `surrogate/outputs/models/registry.json` and checks the SHA-256 fingerprint of every model
file before using it — a changed file is refused. No dataset and no weather file are needed.

In [2]:
t0 = time.perf_counter()
s = Surrogate.load()
print(f"loaded in {time.perf_counter() - t0:.2f} s")
info = s.info()
pd.DataFrame({t: {"model file": info["target_object"][t],
                  "model / features / form": "{model} / {feature_set} / {form}".format(**info["picks"][t]),
                  "unit": info["units"][t]} for t in s.TARGETS}).T

loaded in 0.55 s


,model file,model / features / form,unit
heating_demand_kwh_m2,gpr_heating,gpr / C-G+I / log1p,kWh/m²
cooling_demand_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²
eui_kwh_m2,xgb_eui,xgb / C-G+I / decomposed,kWh/m²
peak_heating_w_m2,mlp_physics,mlp / C-G+I / direct,W/m²
peak_cooling_w_m2,mlp_cg,mlp / C-G / direct,W/m²
zone_cooling_core_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²
zone_cooling_south_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²
zone_cooling_east_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²
zone_cooling_north_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²
zone_cooling_west_kwh_m2,mlp_cg,mlp / C-G / direct,kWh/m²


In [3]:
pd.DataFrame({k: {"file": v["path"], "MB": round(v["bytes"] / 1e6, 2), "sha256": v["sha256"][:12] + "…"}
              for k, v in info["files"].items()}).T

,file,MB,sha256
gpr_heating,models/gpr_heating_slim.pkl,0.42,fef62058a24f…
xgb_eui,models/xgb_eui.pkl,5.63,04df009f0bab…
mlp_cg,models/mlp_cg.pkl,2.29,c31e0ef89e55…
mlp_physics,models/mlp_physics.pkl,2.31,f884cadc9668…
intervals,models/intervals.pkl,9.46,460ad9899ac8…


## 2. Three designs in a training city (Colombo)
A design is a dictionary (or a table) with the 21 inputs of `design_space.yaml`. Below: a mid-range design,
the same building with low-SHGC glass and deep overhangs, and the same building with large high-SHGC windows.

In [4]:
b = s.bounds()
base = {c: float((b.at[c, "min"] + b.at[c, "max"]) / 2) for c in b.index}
base.update(n_floors=3, orientation=0.0)
shaded = dict(base, glazing_shgc=0.25, overhang_depth_n=1.2, overhang_depth_e=1.2,
              overhang_depth_s=1.2, overhang_depth_w=1.2, sidefin_depth=0.6)
glassy = dict(base, wwr_north=0.7, wwr_east=0.7, wwr_south=0.7, wwr_west=0.7, glazing_shgc=0.7,
              overhang_depth_n=0.0, overhang_depth_e=0.0, overhang_depth_s=0.0, overhang_depth_w=0.0,
              sidefin_depth=0.0)
designs = pd.DataFrame([base, shaded, glassy], index=["mid-range", "shaded, low-SHGC", "large windows"])
designs.T

,mid-range,"shaded, low-SHGC",large windows
floor_area,5750.00,5750.00,5750.00
aspect_ratio,2.00,2.00,2.00
n_floors,3.00,3.00,3.00
floor_height,3.75,3.75,3.75
wall_r_value,4.50,4.50,4.50
roof_r_value,7.00,7.00,7.00
floor_r_value,2.75,2.75,2.75
wall_thermal_mass,225.00,225.00,225.00
infiltration_ach,1.05,1.05,1.05
wwr_north,0.45,0.45,0.70


In [5]:
res = s.predict(designs, "colombo")           # default: 90 % ranges
main = {"heating_demand_kwh_m2": "heating", "cooling_demand_kwh_m2": "cooling", "eui_kwh_m2": "EUI"}
table = pd.DataFrame({f"{name} (kWh/m²)": [f"{res.values.at[d, t]:.1f}  ({res.lower.at[d, t]:.1f} – {res.upper.at[d, t]:.1f})"
                                           for d in res.values.index] for t, name in main.items()},
                     index=res.values.index)
print("climate level:", res.climate["level"], "| range type:", res.interval_type)
print(res.message)
print(res.heating_note)
table                                         # prediction (90 % range)

climate level: training | range type: standard
This site is one of the 10 climates the model was trained on. Each range shows where the full EnergyPlus result is expected to fall 9 times out of 10.
Heating predictions hardly respond to shading devices or thermal mass (their simulated effect on heating is below 2 %). Judge shading through cooling and total energy use.


,heating (kWh/m²),cooling (kWh/m²),EUI (kWh/m²)
mid-range,0.5 (0.4 – 0.6),84.3 (83.2 – 85.4),224.4 (221.0 – 227.8)
"shaded, low-SHGC",0.5 (0.4 – 0.6),75.1 (74.1 – 76.0),212.2 (208.8 – 215.7)
large windows,0.5 (0.3 – 0.6),121.0 (117.9 – 124.2),260.0 (257.2 – 262.9)


**Reading it:** each range shows where the full EnergyPlus result is expected to fall 9 times out of 10.
Colombo needs almost no heating, so the cooling and EUI columns are the ones that separate the designs.
The `level` argument gives 50, 80, 90 or 95 % ranges:

In [6]:
pd.DataFrame({f"{int(l * 100)} %": (lambda r: f"{r.lower.at['mid-range', 'eui_kwh_m2']:.1f} – "
                                              f"{r.upper.at['mid-range', 'eui_kwh_m2']:.1f}")(s.predict(designs, "colombo", level=l))
              for l in R.LEVELS}, index=["EUI range, mid-range design (kWh/m²)"])

,50 %,80 %,90 %,95 %
"EUI range, mid-range design (kWh/m²)",222.9 – 225.9,221.7 – 227.0,221.0 – 227.8,220.2 – 228.6


## 3. A new site from its weather file
Any site can be used through its EPW file: the tool computes the same 14 climate values as in training
(`Site.from_epw`). Here London — a site the model never trained on. The tool says how unusual the site is,
which climate values are outside the training range, and switches to the wider **new-site** ranges.

In [7]:
london = Site.from_epw(EPW_DIR / "london.epw", name="London")
r_lon = s.predict(designs, london)
c = r_lon.climate
print(f"level: {c['level']} | distance {c['distance']:.2f} (limit {c['d_max']:.2f}) | nearest: {c['nearest_name']}")
print("outside the training range:", ", ".join(c["out_of_range_readable"]))
print("range type:", r_lon.interval_type)
print()
print(r_lon.message)

level: extrapolation | distance 2.39 (limit 3.56) | nearest: Chicago
outside the training range: cooling degree-days (base 10 °C), annual solar radiation, latitude, humid heat during office hours, mild hours (21-24 °C) during office hours, hot hours (above 24 °C) during office hours
range type: new_site

This site is outside the model's experience (nearest training climate: Chicago; outside the training range for: cooling degree-days (base 10 °C), annual solar radiation, latitude, humid heat during office hours, mild hours (21-24 °C) during office hours, hot hours (above 24 °C) during office hours). Absolute values may be off by 15 % or more - in a test on an unseen cool, low-sun site, cooling was over-predicted by about 14 % and heating under-predicted by about 16 % - but the ranking of designs stayed reliable. Use the results to compare designs, and confirm the chosen design with an EnergyPlus simulation.


In [8]:
r_chi = s.predict(designs, "chicago")
pd.concat({"Chicago (training)": pd.DataFrame({"EUI": r_chi.values["eui_kwh_m2"],
                                               "90 % width": r_chi.upper["eui_kwh_m2"] - r_chi.lower["eui_kwh_m2"]}),
           "London (new site)": pd.DataFrame({"EUI": r_lon.values["eui_kwh_m2"],
                                              "90 % width": r_lon.upper["eui_kwh_m2"] - r_lon.lower["eui_kwh_m2"]})},
          axis=1)

Chicago (training)            London (new site)           
                                EUI 90 % width               EUI 90 % width
mid-range                    110.73       5.07             76.29      43.72
shaded, low-SHGC             106.26       4.59             71.72      41.58
large windows                132.72       6.08             91.65      48.39

The London ranges are much wider than Chicago's: with only 10 training climates, a new site is less certain,
and the tool says so. The ranking of the three designs is what the tool is best at in a new climate.

## 4. A design outside the design space is refused
Nothing outside `design_space.yaml` was ever simulated, so by default (`strict=True`) such a design is refused with
a plain message. `strict=False` predicts anyway and flags the design.

In [9]:
too_big = dict(base, floor_area=20000.0)
try:
    s.predict(too_big, "colombo")
except DesignError as e:
    print("refused:", e)
r = s.predict(pd.DataFrame([base, too_big], index=["ok", "too big"]), "colombo", strict=False)
r.flags

refused: floor_area = 20000 is outside the allowed range 1500-10000.


,design_in_domain,design_violations
ok,True,
too big,False,floor_area


## 5. Optimiser use: 100,000 designs at once
Phase 4 will call `predict_values` (values only, a NumPy array in `Surrogate.TARGETS` order) on whole populations.
`bounds()` gives the search box. Here: 100,000 random designs in Colombo, then the five lowest-EUI designs.

In [10]:
pop = R.lhs_designs(100_000, seed=1)
t0 = time.perf_counter()
Y = s.predict_values(pop, "colombo")
dt = time.perf_counter() - t0
print(f"{len(pop):,} designs in {dt:.1f} s ({len(pop) / dt:,.0f} designs/s; illustrative - see registry_report.md)")
eui = Y[:, s.TARGETS.index("eui_kwh_m2")]
best = pop.assign(eui_kwh_m2=eui).nsmallest(5, "eui_kwh_m2")
best[["eui_kwh_m2", "glazing_shgc", "glazing_u", "wwr_south", "wwr_west", "wall_r_value", "roof_r_value",
      "overhang_depth_s", "n_floors", "aspect_ratio"]]

100,000 designs in 3.1 s (32,106 designs/s; illustrative - see registry_report.md)


,eui_kwh_m2,glazing_shgc,glazing_u,wwr_south,wwr_west,wall_r_value,roof_r_value,overhang_depth_s,n_floors,aspect_ratio
42360,195.29,0.12,2.31,0.31,0.51,5.58,11.00,0.97,4.0,2.93
23112,196.01,0.12,2.37,0.28,0.25,6.81,8.99,1.20,5.0,2.53
43203,196.53,0.10,1.27,0.35,0.66,7.79,7.68,1.32,5.0,2.87
76499,196.63,0.17,0.88,0.13,0.33,4.18,11.46,0.58,5.0,1.25
29761,196.78,0.15,1.85,0.15,0.14,3.14,8.59,1.27,5.0,2.75


These five are only a quick screen, not an optimisation (that is Phase 4). Before a chosen design is reported,
it is re-simulated in EnergyPlus — the design dictionary has exactly the inputs the IDF generator needs.

## 6. Notes for Phase 4 and 5
- **One site per call.** Optimise one climate at a time; the web service shows one site.
- **Speed figures must be measured on a rested Mac** (plugged in, idle ~10 min): right after heavy work the fanless
  MacBook Air ran everything about 2× slower (3.8 step 4).
- The heating model hardly responds to shading and thermal mass (≤ 2 %); judge shading through cooling and EUI.
- `res.to_dict()` gives a JSON-ready answer for the web service; `res.to_frame()` a table for CSV.